In [ ]:
import re
import torch
v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
!pip install --no-deps bitsandbytes==0.45.5 accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
!pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
!pip install --no-deps unsloth
!pip install transformers==4.55.4

In [ ]:
# 만약 설치 중 문제가 발생한다면, 다음 단계를 따라 주세요:

# 1. unsloth와 unsloth_zoo를 완전히 제거
!pip uninstall unsloth unsloth_zoo -y

# 2. 가장 안정적인 방법으로 unsloth_zoo를 먼저 설치
!pip install unsloth_zoo

# 3. 나머지 unsloth를 설치하고, 필요한 종속성(bitsandbytes 등)을 다시 설치
# 이전에 사용했던 스크립트를 재사용하여 모든 종속성을 충족시키도록
!pip install unsloth bitsandbytes==0.45.5 accelerate peft trl transformers datasets

# 4. 주피터 노트북 커널을 반드시 재시작(Restart Kernel)

# 1. 4비트 모델 로드와 성능 검증

## 1-1. 목표

1. 양자화된 모델을 Unsloth를 사용하여 불러온다.
2. 이 모델이 답변을 한글로 충분히 잘 수행할 수 있는지를 확인한다.
3. 한글과 영어 번역에 특화된 데이터셋을 불러온다.
4. 기존의 양자화된 모델에 LoRA 어댑터를 부착하여 학습시킨다.
    - 이때 학습은 SFTTrainer를 사용하여 진행
5. 학습 결과를 확인한다.

## 1-2. 4비트 양자화 모델 로드하기

### 1-2-1. QLoRA(Quantized Low-Rank Adaptation)

- LoRA 기술에 4비트 양자화를 결합한 기술
- 일반적인 GPU에서도 효율적인 파인튜닝이 가능해 짐
1. 양자화
    - VRAM 절약의 핵심
    - 수십억 개의 원본 모델 가중치가 16비트, 32비트 등으로 표현되어 있는 것을 4비트로 압축하여 VRAM에 로드
    - 즉, 모델의 크기를 약 4분의 1, 8분의 1로 줄이는 역할
2. 저랭크 적응(LoRA)
    - 모델의 가중치 행렬은 학습을 통해 변화해야 함. 그런데 가중치가 너무 많음.
    - 심지어, 이 가중치에 대한 변화량은 전체 크기에 비해 훨씬 작은 정보량만으로 표현 할 수 있음.
        - **전체 가중치에 대한 모든 학습을 진행하는 것이 아닌(Frozen),** **훨씬 작은 행령 A와 B의 곱으로 대체 (LoRA Adapter)**
    - 효율적인 학습 → $\Delta W = A \times B$
    - **4비트로 양자회되어 학습 불가능 상태인 원본 가중치(Frozen)** 위에 **아주 작고 학습 가능한 행렬 쌍(LoRA Adapter)**을 덧댐
3. 이중 양자화
    - 4비트 양자화 과정에서 발생하는 오프셋값을 다시 양자화해서 저장

### 1-2-2. MS phi-3 양자화 모델 불러오기

1. `FastLanguageModel`
    - Unsloth가 제공하는 모델 로더
    - 딥러닝 과정에서 사용했던 Pytorch의 `from_pretrained`를 최적화 한 버젼
2. **load_in_4bit:** 4비트 양자화 로딩
    - 기존 32비트 또는 16비트로 저장된 가중치를 4비트로 압축해서 VRAM에 전달
3. **사용할 모델: unsloth/phi-3-mini-4k-instruct-bnb-4bit**
    - MicroSoft가 만든 LLM 모델
    - 약 38억개(3.8B)의 파라미터를 가지고 있음.
    - **unlsoth/:** Unsloth에 의해 `bitsandbytes` 4비트 양자화 형식에 맞게 미리 변환되고 최적화 된 버젼을 의미
        - **bnb-4bit:** 4비트 양자화된 모델
    - **4k:** 컨텍스트 길이가 4096인 긴 대화나 문서 처리 가능
4. **dtype = None**
    - 직접 실수의 표현 크기를 지정하는 대신 None으로 설정
    - Unsloth가 현재 GPU 환경에 가장 효율적인 데이터 타입으로 자동 선택

In [40]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # (시퀀스 길이 2048로 설정)
dtype = None # (Unsloth가 자동으로 최적 타입을 찾도록 None으로 설정)
load_in_4bit = True # 4비트 양자화 로딩 (VRAM 절약 핵심)

# Unsloth가 최적화한 4비트 Phi-3 Mini 모델을 로드합니다.
base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/phi-3-mini-4k-instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

==((====))==  Unsloth 2025.10.7: Fast Mistral patching. Transformers: 4.56.2.
   \\   /|    NVIDIA A100-PCIE-40GB. Num GPUs = 1. Max memory: 39.495 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


## 1-3. 모델 기본 성능 테스트

- 모델이 기본적인 한국어 지시를 이해하는지 확인

### 1-3-1. 간단한 대답 받아보기

- LLM 모델은 정해진 특별한 형식으로 대화를 입력받아야 함.
- 예를들어, `시스템 메시지`, `사용자 질문` `AI 응답` 등등을 프롬프트로 구분하여 전달할 때, 각각의 문장을 구분지어 주어야, 올바르게 답변할 수 있음.
- 이때 `<|system|>` `[INST]` 와 같은 토큰을 붙여서 사용함
1. `tokenizer.apply_chat_template`
    - unsloth로 로드하여, 분리한 토크나이저를 활용
    - 모델이 알아 듣는 문자열로 변환
2. tokenizer: 토큰화 및 텐서 변환
3. `TextStreamer`
    - LLM에게 프롬프트를 입력하여 얻어낸 결과물을 즉시 출력하게 해주는 기능
4. generate
    - 대화 생성
    - 이때, 여타 LLM API를 사용하듯, 출력 방식 제어가 가능하지만 지금은 생략

In [ ]:
from transformers import TextStreamer

# 1. 대화 프롬프트 준비 
messages = [
    {"role": "system", "content": "You are a helpful AI assistant."},
    {"role": "user", "content": "너 누구야? 간결하게."} 
]

# 2. 프롬프트를 모델이 알아듣는 '하나의 문자열'로 변환
    # tokenize=False: 응답 결과를 텍스트로 받음
chat_string = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    tokenize = False 
)

# 3. 이 '문자열'을 토크나이저에 넣어 '텐서(입력값)'로 변환
# 'input_ids'와 'attention_mask'가 자동으로 생성
inputs = tokenizer(
    chat_string, 
    return_tensors="pt"
).to(base_model.device) # 모델이 위치한 디바이스(GPU)로 텐서 이동


# TextStreamer: 출력 토큰이 생성되는 즉시 바로 출력
    # 별도 print문 없이 출력 가능
text_streamer = TextStreamer(tokenizer, skip_prompt = True)

# 4. 추론 실행
# _ : 생성된 토큰 ID들 (사용하지 않음)
_ = base_model.generate(
    **inputs, # inputs 딕셔너리('input_ids', 'attention_mask')를 전달
    streamer = text_streamer,
    max_new_tokens = 128,     # 최대 생성 토큰 수
    # temperature = 0.7,      # 생성 온도
    # top_p = 0.9,            # Top-p 샘플링
    use_cache = True          # 캐시 사용으로 속도 향상
)

이 문장은 한국어 언어로 발표되었습니다. 이 문장은 누구를 찾고 있는 것으로 보입니다. 이 문장은 한국어 언어로 발표되었습니다. 이 문장은 누구를 찾고 ��


# 2. 데이터 전처리

- `SFTTrainer`를 통해 학습 시킬 수 있도록 하기 위해, 학습 대상 모델을 불러오고, 전처리 하는 과정
- 학습 하고자 하는 데이터셋이 우리 모델이 이해할 수 있는 형태와 완전히 일치하지 않을 가능성이 높음
- 따라서, 가지고 있는 데이터 셋을 LLM이 이해할 수 있도록 재구성
- 즉, `입력-정답`으로 구성된 데이터를 LLM이 이해할 수 잇는 `역할` `질문` `모범 답안`의 맥락으로 전달하여 학습하도록 할 것

## 2-1. 원본 데이터 로드

- `lemon-mint/korean_parallel_sentences_v1.1` 데이터 셋을 사용할 것

### 2-1-1. lemon-mint/korean_parallel_sentences_v1.1

1. 한국어-영어 병렬 말뭉치
    - 기게 번역 모델이나 다국어 언어 모델을 훈련하거나 평가하는데 사용
2. 특징
    - 한국어 문장과 그에 대응하는 영어 번역 문장이 쌍으로 구성
    - 약 50만개에 가까운 병렬 문장 쌍을 포함하고 있음.

### 2-1-2. 데이터 불러오기

- 본 실습에서는 LoRA의 효율성을 보기 위하여, 전체 데이터셋을 사용하지는 않을 것임.
- 원활한 실습을 위한 데이터셋 (1000개 혹은 10000개)를 기준으로 학습 진행 예정

In [36]:
from datasets import load_dataset
import pandas as pd

# 1. 데이터셋 로드 (아주 작은 데이터)
    # streaming=True: 메모리 절약용 스트리밍 로드
    # False인 경우: 전체 데이터를 메모리에 올림
    # split='train': 훈련 데이터셋만 로드
dataset = load_dataset("lemon-mint/korean_parallel_sentences_v1.1", split='train', streaming=True)
sample_dataset = dataset.take(20000)  
# iterable_dataset: 데이터를 실제로 로드하지 않고, 필요할 때마다 불러오는 방식
print(sample_dataset)   # 데이터셋 객체만 불러 온 것

# 2. Pandas DataFrame으로 변환해서 구조 확인
print('실제 데이터 로드는 여기서 진행 됩니다.')
sample_list = list(sample_dataset)  # iterable_dataset을 실제 리스트로 변환
df = pd.DataFrame(sample_list)

# 3. 데이터셋 구조 및 내용 출력
print(df.info())
print(f"\n 총 데이터 개수: {len(df)} \n")

print(df.head())

IterableDataset({
    features: ['korean', 'english'],
    num_shards: 1
})
실제 데이터 로드는 여기서 진행 됩니다.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   korean   20000 non-null  object
 1   english  20000 non-null  object
dtypes: object(2)
memory usage: 312.6+ KB
None

 총 데이터 개수: 20000 

                                              korean  \
0         무두족류는 머리가 없고, 대신 몸통에 직접 붙어 있는 발달된 발이 있습니다.   
1  프랑스와 미국의 관계는 오랜 역사를 가지고 있으며, 긴밀한 협력과 때로는 갈등으로 ...   
2             마을 사람들은 마을 축제를 준비하며 마을 거리를 장식하고 있었습니다.   
3  뱀은 길고 가늘며, 비늘로 덮여 있습니다. 뱀은 독이 있거나 독이 없을 수 있습니다...   
4  새 신발은 검은색 가죽으로 만들어졌고, 매우 편안했다. 신발을 신고 밖을 산책하니 ...   

                                             english  
0  Acephala have no head and instead have well-de...  
1  France and the United States have a long and c...  
2  The villagers were decorating the village stre...  
3  Ophidia are elonga

### 2-1-3. ChatML 형식 변환

- Phi-3 모델이 이해할 수 있도록 이 데이터셋의 형식을 변환 할 것
- 현재 데이터 셋은 하나의 row가 korean, english 2개의 column으로 이루어져 있음.

1. list 형태의 데이터를 전처리에 용이하도록 Dataset 객체로 변환
2. LLM이 이해하기 좋도록 messages 형태로 변환
3. 변환 결과 출력

In [37]:
from datasets import Dataset

# 0. 리스트 형태의 데이터를 Hugging Face Dataset 객체로 변환
dataset = Dataset.from_list(sample_list)

# 1. 전처리 함수 정의
def create_chat_prompt(example):
    # Phi-3 (ChatML) 형식에 맞춰 'messages' 리스트 생성
    # 시스템: 역할 부여
    # 입력 (user role): 영어 문장, 
    # 정답 (assistant role): 한국어 문장
    messages = [
        {
            "role": "system",
            "content": "You are an expert translator. Translate the user's English text into Korean."
        },
        {
            "role": "user",
            "content": example["english"] # 입력 = 영어
        },
        {
            "role": "assistant",
            "content": example["korean"] # 정답 = 한국어
        }
    ]
    
    # 'messages' 키를 가진 딕셔너리를 반환
    return {"messages": messages}

# 2. 1000개 데이터 전체에 함수 적용
processed_dataset = dataset.map(
    create_chat_prompt, 
    remove_columns=['korean', 'english'] # 기존 컬럼은 제거
)

# 3. 결과 확인 
from pprint import pprint
pprint(processed_dataset[0])

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

{'messages': [{'content': "You are an expert translator. Translate the user's "
                          'English text into Korean.',
               'role': 'system'},
              {'content': 'Acephala have no head and instead have '
                          'well-developed feet that are directly attached to '
                          'the torso.',
               'role': 'user'},
              {'content': '무두족류는 머리가 없고, 대신 몸통에 직접 붙어 있는 발달된 발이 있습니다.',
               'role': 'assistant'}]}


## 2-2. 전처리 결과 확인 겸 모델 성능 재검증

### 2-2-1. `processed_dataset` 확인 및 대화 생성

1. 테스트할 샘플 준비 ('processed_dataset'의 첫 번째 데이터)
2. '정답(assistant)' 부분은 제외한 프롬프트 구성
3. Phi-3으로 대화 생성
4. 결과: `아침라는 종은 머리를 가지지 않으며, 대담하는 발등이 턱에 연결되어 있다.<|end|>`
    - 정답: 무두족류는 머리가 없고, 대신 몸통에 직접 붙어 있는 발달된 발이 있습니다.
    - 원 정답과 매우 거리가 먼 상태임을 알 수 있음.

In [38]:
from transformers import TextStreamer

# 1. 테스트할 샘플 준비 ('processed_dataset'의 첫 번째 데이터)
test_prompt = processed_dataset[0]['messages']

# 2. '정답(assistant)' 부분은 제외한 프롬프트 구성
test_messages = [
    test_prompt[0], # System 프롬프트 ("You are an expert translator...")
    test_prompt[1]  # User 프롬프트 (영어 문장)
]

# 입력값 텐서 생성
chat_string = tokenizer.apply_chat_template(
    test_messages,
    add_generation_prompt = True,
    tokenize = False 
)

# 토큰화
inputs = tokenizer(
    chat_string, 
    return_tensors="pt"
).to(base_model.device)


text_streamer = TextStreamer(tokenizer, skip_prompt = True)

# 대화 생성
_ = base_model.generate(
    **inputs,
    streamer = text_streamer, 
    max_new_tokens = 128, 
    use_cache = True
)

아침라는 종은 머리를 가지지 않으며, 대담하는 발등이 턱에 연결되어 있다.<|end|>


# 3. LoRA 어댑터 적용

- 현재 `base_model`은 수십억 개의 파라미터를 가진 모델
- 현재 상황에서 바로 파인튜닝을 진행 하는 것(Full Fine-Tuning)은 4비트 양자화가 되어 있어도 엄청난 VRAM을 필요로 할 것
- LoRA의 핵심은 이 거대한 원본 모델 (Phi-3 Mini : 3.8B)은 **전부 동결**시키고 학습시킬 태스크(번역)에 필요한 **작은 어댑터**만 추가로 부착해서, 오직 이 어댑터만 학습 시키는것

## 3-1. Unsloth로 LoRA 어댑터 주입

### 3-1-1. LoRA핵심 파라미터

- 가장 적합한 파라미터 수치는 `실험을 통해 최적화` 필요
1. **r (rank)**
    - LoRA (Low-Rank)의 핵심인 랭크
    - 모든 `target_modules`에 부착될 LoRA 행렬 A와 B의 내부 차원을 정의
        - r이 높을수록 LoRA 어댑터가 가지는 파라미터수가 늘어나는 셈
        - 단, 그만큼 VRAM 사용량이 늘어나고, 학습 속도도 느려짐
2. **lora_alpha**
    - LoRA 어댑터의 출력에 곱해지는 **스케일링 팩터**
    - 학습된 가중치의 영향력을 조절
    - LoRA 학습 속도를 높이거나 모델 성능을 개선하는 데 자주 사용되는 전략
3. **lora_dropout**
    - 훈련 과정에서 LoRA 가중치 행렬의 일부를 임의의 0으로 설정
    - 학습 데이터에 과적합되는 것을 방지하기 위해 사용
    - 0.05는 5% 확률을 의미
4. `target_modules`
    - 어댑터를 부착할 적용 대상을 의미
    - `["q_proj", "k_proj", "v_proj", "o_proj"]`
        - 트랜스포머의 어텐션(Attention) 블록 핵심, QKV (Query, Key, Value) 와 Output 프로젝션 레이어
    - `["gate_proj", "up_proj", "down_proj"]`
        - FFN (Feed-Forward Network) 내부의 선형 레이어

In [ ]:
from unsloth import FastLanguageModel

# 1. LoRA 설정
base_model = FastLanguageModel.get_peft_model(
    base_model, 
    r = 128,
    lora_alpha = 256, # (r의 2배)
    bias = "none",
    # Phi-3 모델에서 LoRA를 적용할 레이어(모듈) 이름
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
)

Unsloth 2025.10.7 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


## 3-2. SFTrainer 설정 및 모델 학습

- `trl` 라이브러리의 `SFTTrainer`를 활용 - 파인 튜닝 과정을 매우 간단하게 만들어주는 도구
- 단, SFTTrainer를 사용하기 위해서는 이전에 만들어 둔 `messages` 리스트를 트레이너가 학습 시킬 수 있도록 가공 작업이 필요

### 3-2-1. 데이터 최종 변환

1. 이전에 작성해 둔 messages는 LLM이 이해할 수 있도록 만든 형태
2. SFTTrainer는 학습을 위해서 이 리스트의 모든 값을 `하나의 문자열` 로 변환하여야 함
3. 변환된 문자열을 다시 `토큰 시퀀스` 로 변환하여, 학습을 위한 데이터로 가공이 필요

- 주의
    - 기존에는 LLM이 이해할 수 있도록 `<|assistant|>` 와 같은 특수 토큰을 추가하였지만, 지금은 학습 과정이므로 이러한 토큰은 불필요함
    - 따라서, `tokenize=False` 설정

In [ ]:
# 'text' 컬럼 생성
def formatting_prompts_func(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize = False, # 주의
        add_generation_prompt = False
    )
    # 'text' 키를 가진 딕셔너리 반환
    return { "text": text }


final_dataset = processed_dataset.map(
    formatting_prompts_func,
    remove_columns=['messages'] # 기존 'messages' 컬럼 제거
)


Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

### 3-2-2. SFTTrainer 및 TrainingArguments 설정

1. **SFTTrainer 설정**
    - model: 사용할 모델 (Phi-3 mini)
    - train_dataset: 학습할 데이터 셋 (final_dataset)
    - dataset_text_field: final_dataset의 `text` 컬럼의 값을 학습용 토큰 시퀀스로 변환
2. **TrainingArguments (학습 전략 설정)**
    - per_device_train_batch_size: 한 번의 스텝에서 학습할 배치 사이즈
    - gradient_accumulation_steps
        - VRAM 최적화의 핵심
        - 가중치 업데이트를 몇 번의 step 이후에 진행할 것인지를 설정
        - 데이터를 batch size에 따라 학습하고, 이때 학습한 기울기를 버리지 않고 누적
        - 지정된 스텝이 누적된 후에야 가중치 업데이트를 진행
    - optim: 옵티마이저 설정
        - 이 실습에서는 `adamw_bnb_8bit`를 사용
            - 모델 가중치와 옵티마이저 상태를 8bit 양자화
    - lr_scheduler_type: 학습 스케쥴러 설정
        - 이 실습에서는 `cosine_with_restarts`를 사용
            - 학습률을 코사인 곡선처럼 주기적으로 변화시키며 진행
            - 도중, 학습률이 0이 되는 경우 다시 처음부터 restart
    - 그외 각종 파라미터는 천천히 찾아보기~

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = base_model,
    tokenizer = tokenizer,
    train_dataset = final_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,  # 처음 모델 로드시 설정하였음.

    args = TrainingArguments(
      per_device_train_batch_size = 16,      # 배치 사이즈 16
      gradient_accumulation_steps = 2,       # 누적 단계 2
      warmup_ratio = 0.1,                    # 워밍업 비율이란: 학습 초기에 학습률을 점진적으로 증가시키는 비율

      num_train_epochs = 3,                  # 에폭수 3
      learning_rate = 1e-4,                  # 0.0001 학습률
      lr_scheduler_type = "cosine_with_restarts",         

      fp16 = not torch.cuda.is_bf16_supported(),  # 16비트 부동소수점 혼합 정밀도 사용
      bf16 = torch.cuda.is_bf16_supported(),      # bfloat16 혼합 정밀도 사용
      logging_steps = 50,                    # 로깅 간격: 몇 스텝마다 학습 상태를 출력할지 설정
      optim = "adamw_bnb_8bit",              
      weight_decay = 0.01,                   # 가중치 감쇠 (오버피팅 방지)
      seed = 42,
      output_dir = "outputs",                # 모델 저장 디렉토리
      report_to = "none",                    # 로깅 리포트 대상 (none: 리포트 안함
  ),
)

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/20000 [00:00<?, ? examples/s]

### 3-2-3. 학습 및 결과 분석

- 학습 시작 전 현재 모델 및 옵티마이저가 차지하는 기본 메모리 출력
- `trainer.train()` : 학습!
- 학습 과정 중 GPU가 최대로 사용한 메모리를 기록
    - 만약, 충분한 메모리가 있다는게 확인된다면, 다음 번 하이퍼파라미터 조절시 유용하게 사용가능

In [44]:
#  GPU 메모리 상태 확인 (학습 전)
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved before training.")

# --- 6. 학습 시작 ---
trainer_stats = trainer.train()

#  GPU 메모리 상태 확인 (학습 후)
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"Peak reserved memory during training = {used_memory} GB.")

GPU = NVIDIA A100-PCIE-40GB. Max memory = 39.495 GB.
7.24 GB of memory reserved before training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 20,000 | Num Epochs = 3 | Total steps = 1,875
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 2 x 1) = 32
 "-____-"     Trainable parameters = 239,075,328 of 4,060,154,880 (5.89% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
50,0.963200
100,0.710700
150,0.658200
200,0.621400
250,0.594400
300,0.575200
350,0.564300
400,0.541000
450,0.536100
500,0.525300


Peak reserved memory during training = 17.148 GB.


# 4. 최종 검증

## 4-1. 기존과 동일한 데이터로 확인하기

In [45]:
# '동일한' 테스트 샘플 준비
test_prompt = processed_dataset[0]['messages']
test_messages = [
    test_prompt[0], # System 프롬프트 ("You are an expert translator...")
    test_prompt[1]  # User 프롬프트 (영어 문장)
]
english_input = test_prompt[1]['content']
korean_answer = test_prompt[2]['content']

chat_string = tokenizer.apply_chat_template(
    test_messages,
    add_generation_prompt = True, 
    tokenize = False
)
inputs = tokenizer(
    chat_string,
    return_tensors="pt"
).to(base_model.device)


print(f"입력: {english_input}")
print(f"정답 예시: {korean_answer}")
print('예측: ')
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = base_model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 128, # 최대 128 토큰 생성
    use_cache = True
)

입력: Acephala have no head and instead have well-developed feet that are directly attached to the torso.
정답 예시: 무두족류는 머리가 없고, 대신 몸통에 직접 붙어 있는 발달된 발이 있습니다.
예측: 
무두추는 머리가 없고, 대신 몸통에 직접 붙어 있는 발근이 잘 발달되어 있습니다.<|end|>


## 4-2. 완전히 새로운 문장으로 확인하기

In [46]:
# 새로운 데이터와 정답 준비
# 단, 프롬프트는 동일하게 유지
test_prompt = processed_dataset[0]['messages']
test_prompt[1] = {'content': 'he rapid advancement of artificial intelligence is fundamentally changing how we approach complex problems in various industries.', 'role': 'user'}
test_messages = [
    test_prompt[0], # System 프롬프트 ("You are an expert translator...")
    test_prompt[1]
]
english_input = 'he rapid advancement of artificial intelligence is fundamentally changing how we approach complex problems in various industries.'
korean_answer = '인공지능의 급속한 발전은 다양한 산업에서 우리가 복잡한 문제에 접근하는 방식을 근본적으로 변화시키고 있습니다.'

chat_string = tokenizer.apply_chat_template(
    test_messages,
    add_generation_prompt = True,
    tokenize = False
)

inputs = tokenizer(
    chat_string,
    return_tensors="pt"
).to(base_model.device)


print(f"입력: {english_input}")
print(f"정답 예시: {korean_answer}")
print('예측: ')
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = base_model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 128, # 최대 128 토큰 생성
    use_cache = True
)

입력: he rapid advancement of artificial intelligence is fundamentally changing how we approach complex problems in various industries.
정답 예시: 인공지능의 급속한 발전은 다양한 산업에서 우리가 복잡한 문제에 접근하는 방식을 근본적으로 변화시키고 있습니다.
예측: 
인공지능의 급속한 발전은 다양한 산업에서 우리가 처리하는 복잡한 문제에 대한 접근 방식을 바꾸고 있습니다.<|end|>
